In [18]:
import json
data = []
for i in range(2):
    with open(f"darija_worker{i}.jsonl","r",encoding='utf-8') as file:
        for line in file:
            # Parse each line as a JSON object
            json_object = json.loads(line.strip())
            data.append(json_object)
len(data)

124853

In [19]:
import pandas as pd
df = pd.DataFrame(data)
df.head()

,id,en,fr,darija
0,0,The Wanderer,Le grand Meaulnes,اللي كيدور
1,1,Alain-Fournier,Alain-Fournier,ألان فورنييه
2,2,First Part,PREMIÈRE PARTIE,الجزء الأول
3,3,I,CHAPITRE PREMIER,أنا
4,4,THE BOARDER,LE PENSIONNAIRE,اللي كاري بيت


In [20]:
df.duplicated(subset="id").sum()

np.int64(5)

In [23]:
df = df.drop_duplicates(subset="id")
df

,id,en,fr,darija
0,0,The Wanderer,Le grand Meaulnes,اللي كيدور
1,1,Alain-Fournier,Alain-Fournier,ألان فورنييه
2,2,First Part,PREMIÈRE PARTIE,الجزء الأول
3,3,I,CHAPITRE PREMIER,أنا
4,4,THE BOARDER,LE PENSIONNAIRE,اللي كاري بيت
...,...,...,...,...
124848,127080,"Therese took the glass, half emptied it, and h...","Thérèse prit le verre, le vida à moitié et le ...",تيريز خدات الكاس، خوات نصو، وعطات لوران اللي ش...
124849,127081,The result was like lightning. The couple fell...,"Ce fut un éclair, Ils tombèrent l'un sur l'aut...",النتيجة كانت بحال البرق. طاح المزوجين واحد فوق...
124850,127082,The mouth of the young woman rested on the sca...,"La bouche de la jeune femme alla heurter, sur ...",فم الشابة بقى محطوط على داك الأثر اللي خلاتو س...
124851,127083,"The corpses lay all night, spread out contorte...",Les cadavres restèrent toute la nuit sur le ca...,الجثث بقاو الليل كامل، ملوين فوق الأرض د بيت ا...


In [21]:
from datasets import load_dataset
ds = load_dataset("Helsinki-NLP/opus_books", "en-fr", split="train")

In [24]:
ids_list = df["id"].to_list()
filtred_ds = ds.filter(
    lambda x: int(x["id"]) in ids_list
)
filtred_ds

Filter:   0%|          | 0/127085 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'translation'],
    num_rows: 124848
})

In [25]:
filtred_ds_list = []
for row in filtred_ds:
    row["translation"]["id"] = int(row["id"])
    filtred_ds_list.append(
        row["translation"]
    )

In [26]:
ds_df = pd.DataFrame(filtred_ds_list)
ds_df.head()

,en,fr,id
0,The Wanderer,Le grand Meaulnes,0
1,Alain-Fournier,Alain-Fournier,1
2,First Part,PREMIÈRE PARTIE,2
3,I,CHAPITRE PREMIER,3
4,THE BOARDER,LE PENSIONNAIRE,4


In [27]:
result = pd.merge(ds_df, df[["id","darija"]], on='id')
result.head()

,en,fr,id,darija
0,The Wanderer,Le grand Meaulnes,0,اللي كيدور
1,Alain-Fournier,Alain-Fournier,1,ألان فورنييه
2,First Part,PREMIÈRE PARTIE,2,الجزء الأول
3,I,CHAPITRE PREMIER,3,أنا
4,THE BOARDER,LE PENSIONNAIRE,4,اللي كاري بيت


In [28]:
result.sample(13)

,en,fr,id,darija
122640,The feet dangled down.,Les pieds tombaient.,124877,الرجلين كانوا مدليين لتحت.
8939,"""Who subscribes?""",-- Quels sont les souscripteurs?,8939,"""شكون هاد الناس اللي كيتبرعو؟"""
69488,Un autre signe qui ne valait rien pour le comt...,Another sign which boded no good to the Conte ...,71725,حاجة أخرى ما كانتش كتعني والو للكونت، هي أنها ...
123443,"Then he distinguished the frame, and little by...","Puis il distingua le cadre, il se calma peu à ...",125680,ومن بعد بان ليه الكادر ديال التصويرة، وشوية بش...
113144,"It was very hot, and not a sound was heard in ...","Il faisait tres chaud, pas un bruit ne venait ...",115381,كانت السخونية بزاف، وما كان حتى شي صوت مسموع ف...
112604,These boy's clothes--this jacket and these bre...,"Ces vetements de garçon, cette veste et cette ...",114841,حوايج الدراري هادو - هاد الجاكيط وهاد السراول ...
40581,"""You are an amiable and charming young man,"" s...",-- Vous êtes un aimable et charmant jeune homm...,40581,"قالت ليه مدام بوناسيو: ""نتا شاب ظريف ومؤدب."""
28963,The room had been fashioned into a small museu...,La chambre avait été transformée en petit musé...,28963,البيت كان داير بحال شي متحف صغير، والحيوط عامر...
92005,He was lashed around the waist to withstand th...,Il s'était amarré à mi-corps pour résister aux...,94242,كان مربوط من خصرُو باش يصمد قدام دوك المواج ال...
37531,But his wound had rendered him too weak to sup...,Mais le blessé était trop faible encore pour s...,37531,ولكن الجرح ديالو خلاه يكون ضعيف بزاف باش يدير ...


In [29]:
from datasets import Dataset
new_dataset = Dataset.from_pandas(result)
new_dataset

Dataset({
    features: ['en', 'fr', 'id', 'darija'],
    num_rows: 124848
})

In [30]:
new_dataset = new_dataset.select_columns(["id","en","fr","darija"])
new_dataset

Dataset({
    features: ['id', 'en', 'fr', 'darija'],
    num_rows: 124848
})

In [ ]:
new_dataset.push_to_hub("abdeljalilELmajjodi/opus_books_en_fr_darija"
    ,private=True,
    token=""
    )

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/125 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/abdeljalilELmajjodi/opus_books_en_fr_darija/commit/ee21746ff8be5c1e4853b1667a79e836dcf9659e', commit_message='Upload dataset', commit_description='', oid='ee21746ff8be5c1e4853b1667a79e836dcf9659e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/abdeljalilELmajjodi/opus_books_en_fr_darija', endpoint='https://huggingface.co', repo_type='dataset', repo_id='abdeljalilELmajjodi/opus_books_en_fr_darija'), pr_revision=None, pr_num=None)